<a href="https://colab.research.google.com/github/BraedynL0530/autocaptcha/blob/master/SuperCoolCaptchaBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics fastapi uvicorn python-multipart easyocr

import multiprocessing
import time
from fastapi import FastAPI, UploadFile, File
from ultralytics import YOLO
import cv2
import numpy as np
import easyocr

app = FastAPI()

model = None
reader = None

def init_models():
    global model, reader
    if model is None:
        model = YOLO('yolo11n.pt')
    if reader is None:
        reader = easyocr.Reader(['en'])

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    init_models()

    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    results = model(img)

    detected_boxes = []
    for box in results[0].boxes:
        cx, cy, w, h = box.xywh.cpu().tolist()[0]
        class_id = int(box.cls.cpu().tolist()[0])
        label = model.names[class_id]

        detected_boxes.append({
            "coords": [cx, cy, w, h],
            "label": label
        })

    ocr_results = reader.readtext(img, detail=0)
    captcha_prompt = " ".join(ocr_results).lower()

    return {
        "prompt": captcha_prompt,
        "boxes": detected_boxes
    }

def run_api():
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000)

if __name__ == '__main__':
    !fuser -k 8000/tcp


    try:
        multiprocessing.set_start_method('spawn')
    except RuntimeError:
        pass

    p = multiprocessing.Process(target=run_api)
    p.start()

    time.sleep(5)


    !ssh -o StrictHostKeyChecking=no -R 80:localhost:8000 serveo.net

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 33.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Forwarding HTTP traffic from https://b66f9cf4381b1e47-35-186-144-238.serveousercontent.com
Tip (1): Create an account to reserve names. Pro removes the warning page: https://console.serveo.net/settings?n=1&src=s